In [ ]:
import pandas as pd

def load_books(csv_file):
    """Load book dataset from CSV file."""
    try:
        books = pd.read_csv(csv_file)
        return books
    except Exception as e:
        print(f"Error loading dataset: {e}")
        return None

def recommend_books(books, Category, top_n=5):
    """Recommend books that match the desired Category."""
    if "Category" not in books.columns:
        raise ValueError("Dataset must contain a 'Category' column.")

    # Case-insensitive matching
    matched = books[books['Category'].str.contains(Category, case=False, na=False)]

    if matched.empty:
        return f"No books found for Category '{Category}'."

    return matched[['Title', 'Authors', 'Category']].head(top_n)

# Example usage:
if __name__ == "__main__":
    # Replace with your dataset file path
    csv_file = "/content/BooksDatasetClean.csv"

    books = load_books(csv_file)
    if books is not None:
        user_category = input("Enter a Category: ")
        recommendations = recommend_books(books, user_category, top_n=10)
        print("\nRecommended Books:")
        print(recommendations.to_string(index=False))


Enter a Category: Thriller

Recommended Books:
                                    Title                                                 Authors                        Category
                      All Around The Town                                  By Clark, Mary Higgins  Fiction , Thrillers , Suspense
Thorns in Eden (Thorns in Eden Series #1)                                     By Boggs, Dallas E.  Fiction , Thrillers , Suspense
                           Dance of Death               By Preston, Douglas J. and Child, Lincoln  Fiction , Thrillers , Suspense
                       One Dangerous Lady                              By Hitchcock, Jane Stanton  Fiction , Thrillers , Suspense
                            The Historian                                   By Kostova, Elizabeth  Fiction , Thrillers , Suspense
                           The Camel Club                                      By Baldacci, David   Fiction , Thrillers , General
               The Hamilton Case: A Novel  

In [ ]:
import pandas as pd

# --------- LOADING DATASETS ---------
def load_dataset(csv_file):
    """Load a dataset from CSV file."""
    try:
        data = pd.read_csv(csv_file)
        return data
    except Exception as e:
        print(f"Error loading dataset {csv_file}: {e}")
        return None

# --------- BOOK RECOMMENDER ---------
def recommend_books(books, category, top_n=10):
    """Recommend books by category."""
    required_cols = {"Title", "Authors", "Category"}
    if not required_cols.issubset(books.columns):
        raise ValueError("Books dataset must contain columns: Title, Authors, Category")

    matched = books[books['Category'].str.contains(category, case=False, na=False)]

    if matched.empty:
        return f"No books found for category '{category}'."

    return matched[['Title', 'Authors', 'Category']].head(top_n)

# --------- MOVIE RECOMMENDER ---------
def recommend_movies(movies, genres, top_n=10):
    """Recommend movies by genre sorted by popularity."""
    required_cols = {"title", "genres", "popularity"}
    if not required_cols.issubset(movies.columns):
        raise ValueError("Movies dataset must contain columns: title, Genre, Popularity")

    matched = movies[movies['genres'].str.contains(genres, case=False, na=False)]

    if matched.empty:
        return f"No movies found for genre '{genres}'."

    top_movies = matched.sort_values(by="popularity", ascending=False).head(top_n)
    return top_movies[['title', 'genres', 'popularity']]

# --------- MAIN PROGRAM ---------
if __name__ == "__main__":
    # Replace with your dataset file paths
    books_csv = "/content/BooksDatasetClean.csv"
    movies_csv = "/content/Movie Dataset.csv"

    books = load_dataset(books_csv)
    movies = load_dataset(movies_csv)

    choice = input("Do you want recommendations for Books or Movies? ").strip().lower()

    if choice == "books" and books is not None:
        category = input("Enter a book category (e.g. Fantasy, Romance, Science Fiction): ")
        recommendations = recommend_books(books, category, top_n=10)
        print("\nRecommended Books:")
        print(recommendations if isinstance(recommendations, str) else recommendations.to_string(index=False))

    elif choice == "movies" and movies is not None:
        genre = input("Enter a movie genre (e.g. Action, Drama, Comedy): ")
        recommendations = recommend_movies(movies, genre, top_n=10)
        print("\nTop 10 Recommended Movies:")
        print(recommendations if isinstance(recommendations, str) else recommendations.to_string(index=False))

    else:
        print("Invalid choice or dataset not loaded.")


Do you want recommendations for Books or Movies? movies
Enter a movie genre (e.g. Action, Drama, Comedy): Romance

Top 10 Recommended Movies:
                                    title                                                     genres  popularity
                                Elemental                Animation, Comedy, Family, Fantasy, Romance    1008.942
                         After Everything                                             Romance, Drama     474.472
                                 My Fault                                             Drama, Romance     361.025
                       The Little Mermaid                        Adventure, Family, Fantasy, Romance     353.543
                         No Hard Feelings                                            Comedy, Romance     298.801
               My Big Fat Greek Wedding 3                                            Romance, Comedy     295.179
Miraculous: Ladybug & Cat Noir, The Movie Animation, Music, Family,

In [ ]:
import pandas as pd
from openai import OpenAI

# --------- LOADING DATASETS ---------
def load_dataset(csv_file):
    """Load a dataset from CSV file."""
    try:
        data = pd.read_csv(csv_file)
        return data
    except Exception as e:
        print(f"Error loading dataset {csv_file}: {e}")
        return None

# --------- BOOK RECOMMENDER ---------
def recommend_books(books, category, top_n=5):
    """Recommend books by category."""
    required_cols = {"Title", "Authors", "Category"}
    if not required_cols.issubset(books.columns):
        raise ValueError("Books dataset must contain columns: Title, Authors, Category")

    matched = books[books['Category'].str.contains(category, case=False, na=False)]

    if matched.empty:
        return []

    return matched[['Title', 'Authors', 'Category']].head(top_n).to_dict(orient="records")

# --------- MOVIE RECOMMENDER ---------
def recommend_movies(movies, genres, top_n=5):
    """Recommend movies by genre sorted by popularity."""
    required_cols = {"title", "genres", "popularity"}
    if not required_cols.issubset(movies.columns):
        raise ValueError("Movies dataset must contain columns: title, genres, popularity")

    matched = movies[movies['genres'].str.contains(genres, case=False, na=False)]

    if matched.empty:
        return []

    top_movies = matched.sort_values(by="popularity", ascending=False).head(top_n)
    return top_movies[['title', 'genres', 'popularity']].to_dict(orient="records")

# --------- OPENAI RECOMMENDER ---------
def creative_inspiration(api_key, plot_idea, books, movies):
    """Send user plot + recommendations to OpenAI and get creative output."""
    client = OpenAI(api_key=api_key)

    prompt = f"""
    The user is facing writer's block. They are interested in:
    - Book Category Inspiration: {books}
    - Movie Genre Inspiration: {movies}

    Their current plot idea is:
    "{plot_idea}"

    Please suggest:
    1. How they can expand on this idea.
    2. What themes, characters, or twists might work well.
    3. Additional book/movie inspirations that fit.
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",  # You can switch to gpt-4.1 or gpt-3.5-turbo if preferred
        messages=[{"role": "user", "content": prompt}],
        max_tokens=500
    )

    return response.choices[0].message.content

# --------- MAIN PROGRAM ---------
if __name__ == "__main__":
    # Replace with your dataset file paths
    books_csv = "/content/BooksDatasetClean.csv"
    movies_csv = "/content/Movie Dataset.csv"

    # Load datasets
    books = load_dataset(books_csv)
    movies = load_dataset(movies_csv)

    # Ask user for preferences
    book_category = input("Enter a book category (e.g. Fantasy, Romance, Science Fiction): ")
    movie_genre = input("Enter a movie genre (e.g. Action, Drama, Comedy): ")

    # Get recommendations
    recommended_books = recommend_books(books, book_category, top_n=5)
    recommended_movies = recommend_movies(movies, movie_genre, top_n=5)

    print("\n📚 Recommended Books:")
    if recommended_books:
        for b in recommended_books:
            print(f"- {b['Title']} by {b['Authors']} ({b['Category']})")
    else:
        print("No books found.")

    print("\n🎬 Recommended Movies:")
    if recommended_movies:
        for m in recommended_movies:
            print(f"- {m['title']} ({m['genres']}, Popularity: {m['popularity']})")
    else:
        print("No movies found.")


Enter a book category (e.g. Fantasy, Romance, Science Fiction): Fantasy
Enter a movie genre (e.g. Action, Drama, Comedy): Comedy

📚 Recommended Books:
- Elvis in the Morning by By Buckley, William F. ( Fiction , Fantasy , Historical)
- White Gold Wielder - Book Three of The Second Chronicles of Thomas Covenant by By Donaldson, Stephen R. ( Fiction , Fantasy , General)
- Daughter of Regals & Other Tales by By Donaldson, Stephen R. ( Fiction , Fantasy , General)
- The Sands of Time: A Hermux Tantamoq Adventure by By Hoeye, Michael ( Young Adult Fiction , Fantasy , General)
- Hope Was Here (Newbery Honor Book) by By Bauer, Joan ( Young Adult Fiction , Fantasy , General)

🎬 Recommended Movies:
- Barbie (Comedy, Adventure, Fantasy, Popularity: 1069.34)
- Elemental (Animation, Comedy, Family, Fantasy, Romance, Popularity: 1008.942)
- Spy Kids: Armageddon (Family, Comedy, Action, Adventure, Popularity: 613.403)
- Teenage Mutant Ninja Turtles: Mutant Mayhem (Animation, Comedy, Action, Populari